# 02 · Statistical analysis

**Question:** Which selected churn differences are large enough to merit attention, and how uncertain are they? This is an observational sample; tests do not identify causal effects.

Preselected comparisons keep the analysis focused. Holm correction controls family-wise error across the categorical tests shown here.

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.multitest import multipletests

ROOT = Path.cwd()
if not (ROOT / 'src').exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
from datapulse.data import load_telco
DATA_DIR = ROOT / 'data' / 'raw'
candidates = [DATA_DIR / 'Telco-Customer-Churn.csv', DATA_DIR / 'WA_Fn-UseC_-Telco-Customer-Churn.csv']
data_path = next((p for p in candidates if p.exists()), None)
if data_path is None: raise FileNotFoundError('Place IBM CSV in data/raw/. See data/raw/README.md')
df = load_telco(data_path)
df['churned'] = (df['Churn'] == 'Yes').astype(int)
print(f'n={len(df):,}; observed churn rate={df.churned.mean():.1%}')

n=7,043; observed churn rate=26.5%


## Continuous variable: tenure
Welch's t-test does not assume equal variances. We also report a bootstrap confidence interval for the difference in means and a standardized effect size. With large samples, p-values can be small for modest effects, so prioritize effect size and uncertainty.

In [2]:
rng = np.random.default_rng(42)
churn = df.loc[df.churned == 1, 'tenure'].dropna().to_numpy()
stay = df.loc[df.churned == 0, 'tenure'].dropna().to_numpy()
observed = churn.mean() - stay.mean()
boot = np.array([rng.choice(churn, len(churn), replace=True).mean() - rng.choice(stay, len(stay), replace=True).mean() for _ in range(3000)])
welch = stats.ttest_ind(churn, stay, equal_var=False)
pooled_sd = np.sqrt(((len(churn)-1)*churn.var(ddof=1) + (len(stay)-1)*stay.var(ddof=1))/(len(churn)+len(stay)-2))
cohens_d = observed / pooled_sd
pd.Series({'churn_mean_months': churn.mean(), 'retained_mean_months': stay.mean(), 'difference_churn_minus_retained': observed, 'bootstrap_95pct_low': np.quantile(boot,.025), 'bootstrap_95pct_high': np.quantile(boot,.975), 'welch_t': welch.statistic, 'welch_p': welch.pvalue, 'cohens_d': cohens_d}).round(4)

churn_mean_months                  17.9791
retained_mean_months               37.5700
difference_churn_minus_retained   -19.5908
bootstrap_95pct_low               -20.6969
bootstrap_95pct_high              -18.4761
welch_t                           -34.8238
welch_p                             0.0000
cohens_d                           -0.8522
dtype: float64

## Categorical variables: churn independence
Pearson chi-square tests assess whether churn proportions differ across levels. We report Cramér's V as a scale-free effect size. Rare levels and multiple testing should temper interpretation.

In [3]:
features = [c for c in ['Contract', 'InternetService', 'PaymentMethod', 'PaperlessBilling'] if c in df]
rows = []
for column in features:
    table = pd.crosstab(df[column].fillna('Missing'), df['Churn'])
    chi2, p_value, dof, _ = stats.chi2_contingency(table)
    n = table.to_numpy().sum(); min_dim = min(table.shape) - 1
    cramers_v = np.sqrt(chi2 / (n * min_dim)) if min_dim > 0 else np.nan
    rows.append({'feature': column, 'chi2': chi2, 'dof': dof, 'p_value': p_value, 'cramers_v': cramers_v, 'levels': table.shape[0]})
results = pd.DataFrame(rows)
if not results.empty:
    results['p_holm'] = multipletests(results.p_value, method='holm')[1]
    display(results.sort_values('p_holm').round(5))
else: print('No selected categorical features found.')

,feature,chi2,dof,p_value,cramers_v,levels,p_holm
0,Contract,1184.59657,2,0.0,0.41012,3,0.0
1,InternetService,732.30959,2,0.0,0.32245,3,0.0
2,PaymentMethod,648.14233,3,0.0,0.30336,4,0.0
3,PaperlessBilling,258.27765,1,0.0,0.19150,2,0.0


## Interpretation and limits
Write a short conclusion with direction, magnitude, uncertainty, and corrected p-value. Avoid statements that a feature *causes* churn. The sample is not randomized and the analyses are unadjusted one-variable comparisons; overlapping customer characteristics may explain patterns. A retention intervention must be evaluated through a prospective experiment.

Optional extension: estimate adjusted associations with a prespecified logistic regression and bootstrap intervals, while clearly separating explanatory and predictive goals.